In [3]:
import pandas as pd

In [4]:
ml_movies = pd.read_csv("../data/movielens/ratings.csv")
ml_movies.shape
#around 160 reviews per user (I have ~250)

(32000204, 4)

In [5]:
ml_movies = pd.read_csv("../data/movielens/ml_movies_with_tmdbId.csv")
ml_movies.head()

,movieId,title,genres,imdbId,tmdbId
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844.0
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357.0
4,5,Father of the Bride Part II (1995),Comedy,113041,11862.0


In [6]:
ml_movies.shape

(87585, 5)

### Enrich movielens data

In [7]:
# LOCATION: notebooks/02_feature_engineering.ipynb

import os
import time
import requests
import pandas as pd

from dotenv import load_dotenv

load_dotenv("../.env")

TMDB_TOKEN = os.getenv("TMDB_TOKEN")

HEADERS = {
    "Authorization": f"Bearer {TMDB_TOKEN}",
    "accept": "application/json"
}

In [8]:
# LOCATION: notebooks/02_feature_engineering.ipynb


def get_tmdb_features(tmdb_id):

    if pd.isna(tmdb_id):
        return None

    tmdb_id = int(tmdb_id)

    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"

    params = {
        "language": "en-US",
        "append_to_response": "credits,keywords"
    }

    try:
        response = requests.get(
            url,
            headers=HEADERS,
            params=params,
            timeout=15
        )

        response.raise_for_status()

        movie = response.json()

        # -----------------------------
        # Genres
        # -----------------------------

        genres = [
            genre["name"]
            for genre in movie.get("genres", [])
        ]


        # -----------------------------
        # Director
        # -----------------------------

        crew = movie.get(
            "credits", {}
        ).get("crew", [])

        directors = [
            person["name"]
            for person in crew
            if person.get("job") == "Director"
        ]


        # -----------------------------
        # Top cast
        # -----------------------------

        cast = movie.get(
            "credits", {}
        ).get("cast", [])

        top_cast = [
            person["name"]
            for person in cast[:5]
        ]


        # -----------------------------
        # Keywords
        # -----------------------------

        keywords = [
            keyword["name"]
            for keyword in movie.get(
                "keywords", {}
            ).get("keywords", [])
        ]


        # -----------------------------
        # Poster
        # -----------------------------

        poster_path = movie.get("poster_path")

        poster_url = (
            f"https://image.tmdb.org/t/p/w500{poster_path}"
            if poster_path
            else None
        )


        return {
            "tmdb_overview": movie.get("overview"),
            "tmdb_genres": "|".join(genres),
            "director": "|".join(directors),
            "cast": "|".join(top_cast),
            "keywords": "|".join(keywords),
            "release_date": movie.get("release_date"),
            "runtime": movie.get("runtime"),
            "popularity": movie.get("popularity"),
            "vote_average": movie.get("vote_average"),
            "vote_count": movie.get("vote_count"),
            "poster_url": poster_url
        }

    except requests.RequestException as e:

        print(
            f"TMDB error for {tmdb_id}: {e}"
        )

        return None

In [10]:

movie_features = []

total = len(ml_movies)

for i, tmdb_id in enumerate(
    ml_movies["tmdbId"]
):

    features = get_tmdb_features(
        tmdb_id
    )

    movie_features.append(
        features or {}
    )

    if (i + 1) % 100 == 0:
        print(
            f"{i + 1:,} / {total:,} movies enriched"
        )

    # Small delay to be polite to API
    time.sleep(0.03)

100 / 87,585 movies enriched
200 / 87,585 movies enriched
300 / 87,585 movies enriched
400 / 87,585 movies enriched
500 / 87,585 movies enriched
TMDB error for 538286: 404 Client Error: Not Found for url: https://api.themoviedb.org/3/movie/538286?language=en-US&append_to_response=credits%2Ckeywords
600 / 87,585 movies enriched
700 / 87,585 movies enriched
800 / 87,585 movies enriched
900 / 87,585 movies enriched
1,000 / 87,585 movies enriched
1,100 / 87,585 movies enriched
1,200 / 87,585 movies enriched
1,300 / 87,585 movies enriched
1,400 / 87,585 movies enriched
1,500 / 87,585 movies enriched
1,600 / 87,585 movies enriched
1,700 / 87,585 movies enriched
1,800 / 87,585 movies enriched
1,900 / 87,585 movies enriched
2,000 / 87,585 movies enriched
2,100 / 87,585 movies enriched
2,200 / 87,585 movies enriched
2,300 / 87,585 movies enriched
2,400 / 87,585 movies enriched
2,500 / 87,585 movies enriched
2,600 / 87,585 movies enriched
2,700 / 87,585 movies enriched
2,800 / 87,585 movies enri

In [14]:
tmdb_features = pd.DataFrame(
    movie_features
)

movies_enriched = pd.concat(
    [
        ml_movies.reset_index(drop=True),
        tmdb_features
    ],
    axis=1
)

movies_enriched.head()

,movieId,title,genres,imdbId,tmdbId,tmdb_overview,tmdb_genres,director,cast,keywords,release_date,runtime,popularity,vote_average,vote_count,poster_url
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,"Led by Woody, Andy's toys live happily in his ...",Family|Comedy|Animation|Adventure,John Lasseter,Tom Hanks|Tim Allen|Don Rickles|Jim Varney|Wal...,rescue|friendship|mission|jealousy|villain|bul...,1995-11-22,81.0,41.2575,7.989,20565.0,https://image.tmdb.org/t/p/w500/uXDfjJbdP4ijW5...
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844.0,When siblings Judy and Peter discover an encha...,Adventure|Fantasy|Family,Joe Johnston,Robin Williams|Kirsten Dunst|Bradley Pierce|Bo...,based on novel or book|giant insect|board game...,1995-12-15,104.0,4.2650,7.249,11586.0,https://image.tmdb.org/t/p/w500/bdHG5Mo83VPobe...
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602.0,A family wedding reignites the ancient feud be...,Romance|Comedy,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret|Sophia ...,fishing|sequel|old man|best friend|wedding|ita...,1995-12-22,101.0,5.3009,6.466,444.0,https://image.tmdb.org/t/p/w500/1FSXpj5e8l4KH6...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357.0,"Cheated on, mistreated and stepped on, the wom...",Comedy|Drama|Romance,Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine|...,based on novel or book|single mother|divorce|a...,1995-12-22,127.0,4.6529,6.200,213.0,https://image.tmdb.org/t/p/w500/4wjGMwPsdlvi02...
4,5,Father of the Bride Part II (1995),Comedy,113041,11862.0,Just when George Banks has recovered from his ...,Comedy|Family,Charles Shyer,Steve Martin|Diane Keaton|Martin Short|Kimberl...,daughter|baby|parent child relationship|midlif...,1995-12-08,106.0,6.4058,6.272,841.0,https://image.tmdb.org/t/p/w500/rj4LBtwQ0uGrpB...


In [12]:
tmdb_features = pd.DataFrame(
    movie_features
)

In [15]:
movies_enriched.head()

,movieId,title,genres,imdbId,tmdbId,tmdb_overview,tmdb_genres,director,cast,keywords,release_date,runtime,popularity,vote_average,vote_count,poster_url
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,"Led by Woody, Andy's toys live happily in his ...",Family|Comedy|Animation|Adventure,John Lasseter,Tom Hanks|Tim Allen|Don Rickles|Jim Varney|Wal...,rescue|friendship|mission|jealousy|villain|bul...,1995-11-22,81.0,41.2575,7.989,20565.0,https://image.tmdb.org/t/p/w500/uXDfjJbdP4ijW5...
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844.0,When siblings Judy and Peter discover an encha...,Adventure|Fantasy|Family,Joe Johnston,Robin Williams|Kirsten Dunst|Bradley Pierce|Bo...,based on novel or book|giant insect|board game...,1995-12-15,104.0,4.2650,7.249,11586.0,https://image.tmdb.org/t/p/w500/bdHG5Mo83VPobe...
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602.0,A family wedding reignites the ancient feud be...,Romance|Comedy,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret|Sophia ...,fishing|sequel|old man|best friend|wedding|ita...,1995-12-22,101.0,5.3009,6.466,444.0,https://image.tmdb.org/t/p/w500/1FSXpj5e8l4KH6...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357.0,"Cheated on, mistreated and stepped on, the wom...",Comedy|Drama|Romance,Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine|...,based on novel or book|single mother|divorce|a...,1995-12-22,127.0,4.6529,6.200,213.0,https://image.tmdb.org/t/p/w500/4wjGMwPsdlvi02...
4,5,Father of the Bride Part II (1995),Comedy,113041,11862.0,Just when George Banks has recovered from his ...,Comedy|Family,Charles Shyer,Steve Martin|Diane Keaton|Martin Short|Kimberl...,daughter|baby|parent child relationship|midlif...,1995-12-08,106.0,6.4058,6.272,841.0,https://image.tmdb.org/t/p/w500/rj4LBtwQ0uGrpB...


In [16]:
movies_enriched.to_parquet(
    "../data/movie_with_features.parquet",
    index=False
)

In [18]:
movies_enriched.head()

,movieId,title,genres,imdbId,tmdbId,tmdb_overview,tmdb_genres,director,cast,keywords,release_date,runtime,popularity,vote_average,vote_count,poster_url
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,"Led by Woody, Andy's toys live happily in his ...",Family|Comedy|Animation|Adventure,John Lasseter,Tom Hanks|Tim Allen|Don Rickles|Jim Varney|Wal...,rescue|friendship|mission|jealousy|villain|bul...,1995-11-22,81.0,41.2575,7.989,20565.0,https://image.tmdb.org/t/p/w500/uXDfjJbdP4ijW5...
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844.0,When siblings Judy and Peter discover an encha...,Adventure|Fantasy|Family,Joe Johnston,Robin Williams|Kirsten Dunst|Bradley Pierce|Bo...,based on novel or book|giant insect|board game...,1995-12-15,104.0,4.2650,7.249,11586.0,https://image.tmdb.org/t/p/w500/bdHG5Mo83VPobe...
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602.0,A family wedding reignites the ancient feud be...,Romance|Comedy,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret|Sophia ...,fishing|sequel|old man|best friend|wedding|ita...,1995-12-22,101.0,5.3009,6.466,444.0,https://image.tmdb.org/t/p/w500/1FSXpj5e8l4KH6...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357.0,"Cheated on, mistreated and stepped on, the wom...",Comedy|Drama|Romance,Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine|...,based on novel or book|single mother|divorce|a...,1995-12-22,127.0,4.6529,6.200,213.0,https://image.tmdb.org/t/p/w500/4wjGMwPsdlvi02...
4,5,Father of the Bride Part II (1995),Comedy,113041,11862.0,Just when George Banks has recovered from his ...,Comedy|Family,Charles Shyer,Steve Martin|Diane Keaton|Martin Short|Kimberl...,daughter|baby|parent child relationship|midlif...,1995-12-08,106.0,6.4058,6.272,841.0,https://image.tmdb.org/t/p/w500/rj4LBtwQ0uGrpB...


In [19]:
movies_enriched.shape

(87585, 16)